# **Anticipating Payroll Surprises: A Statistical Learning Framework**

The U.S. Employment Situation Report is one of the most important macroeconomic releases for financial markets. Treasury yields, equity indices, foreign exchange markets, and interest rate expectations can all move sharply within seconds of the report's release. However, market reactions are driven less by the payroll number itself than by how that number compares to expectations.

This project seeks to determine whether publicly available economic and financial market data can be used to anticipate labor market surprises before they occur. Using labor market indicators from the Bureau of Labor Statistics (BLS) and market data from the Federal Reserve Economic Data (FRED) database, I construct a model-based estimate of expected payroll growth and identify periods in which actual payroll growth meaningfully exceeded or fell short of those expectations.

The resulting surprise measures are then used to train binary and multinomial classification models to identify meaningful surprises in future payroll reports. The objective is not simply to forecast payroll growth, but to identify the economic conditions most associated with labor market outcomes that have the potential to move financial markets.

By focusing on surprises rather than headline payroll figures, this analysis mirrors the way macroeconomic information is incorporated into asset prices and provides a practical application of econometrics, feature engineering, and machine learning within a financial markets framework.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv; load_dotenv('C:\git-repository\Blog\.gitignore\.env')
import requests
import xgboost as xgb
from pandas_datareader import data as pdr 
from sklearn.model_selection import (TimeSeriesSplit, RandomizedSearchCV, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    ConfusionMatrixDisplay
)

## **FRED Data Calls**

I have selected the following variables from the St Louis Federal Reserve because they capture key dimensions of labor and financial market conditions, growth expectations, and consumer sentiments that I believe may be influential to labor market surprises. 

Due to the limits of some of my JOLTS variables, I must limit my dataset to observations from 2000 - 2025, which is far from ideal for a macroeconomic time series.

- **PAYEMS** – Total Nonfarm Payroll Employment. Measures the total number of paid workers in the U.S. economy excluding farm employees, private household employees, and nonprofit workers. This serves as the primary target variable for the project.

- **UNRATE** – Civilian Unemployment Rate. Measures the percentage of the labor force that is unemployed and actively seeking work. Rising unemployment rates often signal weakening labor market conditions.

- **ICSA** – Initial Claims for Unemployment Insurance. Measures the number of new unemployment benefit claims filed each week. Initial claims are considered a leading indicator of labor market deterioration.

- **CCSA** – Continuing Claims for Unemployment Insurance. Measures the number of individuals continuing to receive unemployment benefits. Continuing claims provide insight into the persistence of unemployment and the ability of displaced workers to find new employment.

- **DGS2** – 2-Year Treasury Constant Maturity Yield. Represents market expectations for short-term interest rates and Federal Reserve policy. The 2-year yield is highly sensitive to changes in economic outlook and labor market expectations.

- **DGS10** – 10-Year Treasury Constant Maturity Yield. Represents longer-term growth and inflation expectations. Changes in the 10-year yield often reflect shifts in investor sentiment regarding the broader economic outlook.

- **T10Y2Y** – 10-Year Minus 2-Year Treasury Yield Spread. Measures the slope of the Treasury yield curve. Yield curve inversions have historically been associated with slowing economic growth and elevated recession risk.

- **UMCSENT** – University of Michigan Consumer Sentiment Index. Measures consumer confidence regarding current and future economic conditions. Consumer sentiment can provide insight into future spending, hiring, and overall economic activity.

In [ ]:
series = [
    'PAYEMS',
    'UNRATE',
    'ICSA',
    'CCSA',
    'DGS2',
    'DGS10',
    'T10Y2Y',
    'UMCSENT'
    ]

start = '2000-01-01'
end = '2025-01-01'

df = pdr.DataReader(
    series,
    'fred',
    start,
    end
)

fred_df = df.resample('M').mean()
fred_df = fred_df.ffill()

In [ ]:
fred_df = fred_df.rename(
    columns={
        'PAYEMS': 'payrolls', 
        'UNRATE': 'unemployment_rate', 
        'ICSA': 'initial_claims', 
        'CCSA': 'cont_claims', 
        'DGS2': '2yr_yield', 
        'DGS10': '10yr_yield',
        'T10Y2Y': '10yr_2yr_spread',
        'UMCSENT': 'sentiment'
        }).reset_index()

In [ ]:
fred_df['month'] = fred_df['DATE'].dt.month
fred_df['year'] = fred_df['DATE'].dt.year

fred_df.head()

## **BLS Data Calls**

The JOLTS variables were selected because they capture both labor demand and labor market turnover. Together, they provide insight into hiring activity, worker confidence, and labor market deterioration, all of which may contain predictive information regarding future payroll growth and employment report surprises.

- **Job Openings Rate (JTS000000000000000JOR)** –  Measures the number of job openings as a percentage of total employment plus job openings. This serves as a measure of labor demand and reflects employers' willingness to hire additional workers. Rising job opening rates generally indicate a strong labor market.

- **Hires Rate (JTS000000000000000HIR)** – Measures the number of hires during the month as a percentage of employment. The hires rate provides insight into the pace at which firms are actively adding workers and expanding payrolls.

- **Quits Rate (JTS000000000000000QUR)** – Measures the number of voluntary separations as a percentage of employment. The quits rate is often viewed as a measure of worker confidence, as employees are generally more willing to leave their jobs when alternative employment opportunities are readily available.

- **Layoffs and Discharges Rate (JTS000000000000000LDR)** – Measures involuntary separations initiated by employers as a percentage of employment. Rising layoffs and discharges typically indicate weakening labor market conditions and can serve as an early signal of economic deterioration.

In [ ]:
import json
import prettytable
bls_key = os.getenv('BLS')

In [ ]:
BLS_ENDPOINT = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

def fetch_bls_series(series, **kwargs):
    if len(series) < 1 or len(series) > 25:
        raise ValueError("Must pass in between 1 and 25 series ids")
    headers = {'Content-Type': 'application/json'}
    payload = {
        'seriesid': series,
        'registrationKey': bls_key,
    }
    payload.update(kwargs)
    payload = json.dumps(payload)
    response = requests.post(BLS_ENDPOINT, data=payload, headers=headers)
    response.raise_for_status()
    result = json.loads(response.text)
    if result['status'] != 'REQUEST_SUCCEEDED':
        raise Exception(result['message'][0])
    return result

In [ ]:
series = ['JTS000000000000000JOR', 'JTS000000000000000HIR', 'JTS000000000000000QUR', 'JTS000000000000000LDR']
start_year = 1999
end_year = 2026
json_data = fetch_bls_series(series, startyear=start_year, endyear=end_year)

try:
    dfs = []

    for series in json_data['Results']['series']:
        df_initial = pd.DataFrame(series)
        series_col = df_initial['seriesID'][0]

        for i in range(0, len(df_initial) - 1):
            df_row = pd.DataFrame(df_initial['data'][i])
            df_row['seriesID'] = series_col

            if 'code' not in str(df_row['footnotes']): 
                df_row['footnotes'] = ''
            else:
                df_row['footnotes'] = str(df_row['footnotes']).split("'code': '", 1)[1][:1]

            dfs.append(df_row)

    df = pd.concat(dfs, ignore_index=True)

    df.to_csv('blsdata.csv', index=False)

except Exception as e:
    json_data['status'] == 'REQUEST_NOT_PROCESSED'
    print('BLS API has given the following Response:', json_data['status'])
    print('Reason:', json_data['message'])
    print('Error:', str(e))

In [ ]:
openings_01_09_df = (
    df[df['seriesID'] == 'JTS000000000000000JOR']
    .set_index('periodName')[['value', 'year']]
    .rename(columns={'value':'job openings rate'})
)

hires_01_09_df = (
    df[df['seriesID'] == 'JTS000000000000000HIR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'hires rate'})
)

quit_01_09_df = (
    df[df['seriesID'] == 'JTS000000000000000QUR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'quit rate'})
)

discharge_01_09_df = (
    df[df['seriesID'] == 'JTS000000000000000LDR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'discharge rate'})
)

bls_01_09_df = (
    openings_01_09_df
    .join(hires_01_09_df, how='outer')
    .join(quit_01_09_df, how='outer')
    .join(discharge_01_09_df, how='outer')
)

print(bls_01_09_df['year'].unique())

In [ ]:
series = ['JTS000000000000000JOR', 'JTS000000000000000HIR', 'JTS000000000000000QUR', 'JTS000000000000000LDR']
start_year = 2019
end_year = 2026
json_data = fetch_bls_series(series, startyear=start_year, endyear=end_year)

try:
    dfs = []

    for series in json_data['Results']['series']:
        df_initial = pd.DataFrame(series)
        series_col = df_initial['seriesID'][0]

        for i in range(0, len(df_initial) - 1):
            df_row = pd.DataFrame(df_initial['data'][i])
            df_row['seriesID'] = series_col

            if 'code' not in str(df_row['footnotes']): 
                df_row['footnotes'] = ''
            else:
                df_row['footnotes'] = str(df_row['footnotes']).split("'code': '", 1)[1][:1]
    
            dfs.append(df_row)

    df = pd.concat(dfs, ignore_index=True)

    df.to_csv('blsdata.csv', index=False)

except Exception as e:
    json_data['status'] == 'REQUEST_NOT_PROCESSED'
    print('BLS API has given the following Response:', json_data['status'])
    print('Reason:', json_data['message'])
    print('Error:', str(e))

In [ ]:
openings_19_26_df = (
    df[df['seriesID'] == 'JTS000000000000000JOR']
    .set_index('periodName')[['value', 'year']]
    .rename(columns={'value':'job openings rate'})
)

hires_19_26_df = (
    df[df['seriesID'] == 'JTS000000000000000HIR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'hires rate'})
)

quit_19_26_df = (
    df[df['seriesID'] == 'JTS000000000000000QUR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'quit rate'})
)

discharge_19_26_df = (
    df[df['seriesID'] == 'JTS000000000000000LDR']
    .set_index('periodName')[['value']]
    .rename(columns={'value':'discharge rate'})
)

bls_19_26_df = (
    openings_19_26_df
    .join(hires_19_26_df, how='outer')
    .join(quit_19_26_df, how='outer')
    .join(discharge_19_26_df, how='outer')
)
print(bls_19_26_df['year'].unique())

In [ ]:
bls_df = pd.concat([bls_01_09_df, bls_19_26_df]).reset_index()

print(bls_df.dtypes)
bls_df.head()

In [ ]:
bls_df['month_year'] = bls_df['periodName'] + ' ' + bls_df['year'].astype(str)

num_cols = [
    'job openings rate',
    'hires rate',
    'quit rate',
    'discharge rate'
    ]
bls_df[num_cols] = bls_df[num_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

In [ ]:
bls_df = bls_df.groupby(
    'month_year'
    ).agg(
        job_opening_rate = ('job openings rate', 'mean'),
        hire_rate = ('hires rate', 'mean'),
        quit_rate = ('quit rate', 'mean'),
        discharge_rate = ('discharge rate', 'mean')
    ).reset_index()
print(bls_df.dtypes)

In [ ]:
bls_df['month_year'] = pd.to_datetime(bls_df['month_year'])

bls_df['month'] = bls_df['month_year'].dt.month
bls_df['year'] = bls_df['month_year'].dt.year

In [ ]:
bls_df['month']
bls_df = bls_df.sort_values('month_year')#.reset_index()

In [ ]:
bls_df.head()

## **Combining FRED and BLS Datasets**

In [ ]:
master_df = pd.merge(
    fred_df,
    bls_df,
    on=['year', 'month'],
    how='inner'
)

master_df.head()    

## **Feature Engineering**

### Engineered FRED Features

Rolling averages were used to reduce short-term noise and highlight underlying labor market trends, while momentum features were designed to capture accelerating or decelerating changes in economic conditions that may precede payroll report surprises.

- **unemployment_rate_3mo_avg** – Three-month rolling average of the unemployment rate. This feature smooths month-to-month volatility and captures the underlying trend in labor market conditions.

- **initial_claims_lag1** – Previous month's initial unemployment claims. Lagged claims data provide information about labor market conditions that would have been available prior to the employment report release.

- **initial_claims_momentum** – Change in initial unemployment claims over a specified period. This feature measures whether layoffs are accelerating or decelerating.

- **cont_claims_lag1** – Previous month's continuing unemployment claims. Continuing claims provide insight into the persistence of unemployment and the ability of displaced workers to find new employment.

- **cont_claims_momentum** – Change in continuing claims over a specified period. Rising continuing claims may indicate a weakening labor market.

- **2yr_yield_volatility** – Rolling volatility of the 2-Year Treasury yield. This feature captures uncertainty surrounding Federal Reserve policy expectations and short-term economic outlook.

- **10yr_yield_volatility** – Rolling volatility of the 10-Year Treasury yield. Higher volatility may reflect uncertainty regarding long-term growth and inflation expectations.

- **jor_3mo_avg** – Three-month rolling average of the Job Openings Rate. This smooths short-term fluctuations and provides a measure of underlying labor demand.

- **jor_momentum** – Change in the Job Openings Rate over a specified period. Positive momentum suggests strengthening labor demand, while negative momentum may indicate labor market softening.

- **hire_3mo_avg** – Three-month rolling average of the Hires Rate. This feature captures the underlying pace of hiring activity across the economy.

- **hire_momentum** – Change in the Hires Rate over a specified period. Rising hiring momentum may precede stronger payroll growth.

- **quit_3mo_avg** – Three-month rolling average of the Quits Rate. The quits rate is commonly viewed as a measure of worker confidence and labor market strength.

- **quit_momentum** – Change in the Quits Rate over a specified period. Increasing quits may indicate greater confidence among workers and a tighter labor market.

- **discharge_3mo_avg** – Three-month rolling average of the Layoffs and Discharges Rate. This feature provides a smoothed measure of labor market deterioration.

- **discharge_momentum** – Change in the Layoffs and Discharges Rate over a specified period. Rising layoffs may signal weakening labor market conditions and increase the probability of a downside payroll surprise.

In [ ]:
# FRED features

master_df['sentiment_lag1'] = master_df['sentiment'].shift(1)
master_df['payrolls_lag1'] = master_df['payrolls'].shift(1)
master_df['payrolls_change'] = master_df['payrolls'].diff()
master_df['unemployment_rate_3mo_avg'] = master_df['unemployment_rate'].rolling(3).mean()
master_df['initial_claims_lag1'] = master_df['initial_claims'].shift(1)
master_df['initial_claims_momentum'] = master_df['initial_claims'].diff(3)
master_df['cont_claims_lag1'] = master_df['cont_claims'].shift(1)
master_df['cont_claims_momentum'] = master_df['cont_claims'].diff(3)
master_df['2yr_yield_volatility'] = master_df['2yr_yield'].rolling(3).std()
master_df['10yr_yield_volatility'] = master_df['10yr_yield'].rolling(3).std()

### Engineered JOLTS Features

To reduce short-term noise and better capture underlying labor market trends, rolling averages and momentum measures were calculated for each JOLTS variable. Rolling averages provide a smoother view of labor market conditions, while momentum measures capture whether labor demand and labor turnover are strengthening or weakening over time.

- **jor_3mo_avg** – Three-month rolling average of the Job Openings Rate. This feature smooths short-term fluctuations and provides a measure of underlying labor demand. Higher values generally indicate that employers are actively seeking workers and may precede stronger payroll growth.

- **jor_momentum** – Three-month change in the Job Openings Rate. Positive momentum indicates strengthening labor demand, while negative momentum may signal a cooling labor market.

- **hire_3mo_avg** – Three-month rolling average of the Hires Rate. This feature captures the underlying pace of hiring activity while reducing month-to-month noise in the data.

- **hire_momentum** – Three-month change in the Hires Rate. Rising hiring momentum suggests firms are increasing employment at a faster pace, which may contribute to stronger payroll growth.

- **quit_3mo_avg** – Three-month rolling average of the Quits Rate. The quits rate is often viewed as a measure of worker confidence, as employees are more likely to voluntarily leave jobs when alternative opportunities are readily available.

- **quit_momentum** – Three-month change in the Quits Rate. Increasing quits may indicate a tightening labor market and growing confidence among workers.

- **discharge_3mo_avg** – Three-month rolling average of the Layoffs and Discharges Rate. This feature provides a smoothed measure of labor market deterioration and employer-driven separations.

- **discharge_momentum** – Three-month change in the Layoffs and Discharges Rate. Rising layoffs and discharges may signal weakening labor market conditions and increase the likelihood of weaker-than-expected payroll growth.

In [ ]:
# JOLTS Features

master_df['jor_3mo_avg'] = master_df['job_opening_rate'].rolling(3).mean()
master_df['jor_momentum'] = master_df['job_opening_rate'].diff(3)
master_df['hire_3mo_avg'] = master_df['hire_rate'].rolling(3).mean()
master_df['hire_momentum'] = master_df['hire_rate'].diff(3)
master_df['quit_3mo_avg'] = master_df['quit_rate'].rolling(3).mean()
master_df['quit_momentum'] = master_df['quit_rate'].diff(3)
master_df['discharge_3mo_avg'] = master_df['discharge_rate'].rolling(3).mean()
master_df['discharge_momentum'] = master_df['discharge_rate'].diff(3)

In [ ]:
master_df = master_df.dropna().reset_index(drop=True)
print(master_df.columns)

## **Constructing a Proxy for Consensus Forecasts**

Financial markets react not to the headline nonfarm payroll figure itself, but to the difference between the reported value and market expectations. Professional investors typically rely on consensus forecasts compiled from surveys of economists and market participants. However, high-quality consensus forecast data is generally proprietary and not freely available.

To address this limitation, I opted to construct a proxy expectations model using publically available macroeconomic data. A Ridge Regression model was chosen to generate monthly payroll forecasts because it provides a transparent and interpretable framework while mitigating overfitting and multicollinearity. 

The resulting forecast serves as a proxy for market expectations. Labor report surprises are calculated as the difference between the actual change in payrolls and the expected change in payrolls as predicted by the model. The surprise will then be recorded by dummy variables that reflect the direction of the surprise. 

I initially chose the following variables to model market expectations:

- **Unemployment Rate**
- **Initial Unemployment Claims**
- **10 Year Treasury Yield**
- **Job Openings Rate**
- **Hiring Rate**

Prior to model estimation, each variable was examined through histograms and skewness statistics to identify potential departures from normality and determine whether transformations could improve model performance. Variables exhibiting substantial positive skewness were transformed to reduce the influence of extreme observations and stabilize their distributions.

Based on this analysis, a logarithmic transformation was applied to the Initial Unemployment Claims and the Unemployment Rate variables, and a polynomial tranformation was applied to the Hiring Rate variable.

After performing the transformations I proceeded to split the data into chronologically ordered training and testing sets.

In [ ]:
features_l2 = [
    'unemployment_rate',
    'initial_claims',
    '10yr_yield',
    'job_opening_rate',
    'hire_rate'
]

print(master_df[features_l2].skew().sort_values(ascending=False))

master_df[features_l2].hist(bins=20, figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
master_df['log_initial_claims'] = np.log(master_df['initial_claims'])
master_df['hire_rate_squared'] = master_df['hire_rate'] ** 2
master_df['unemployment_rate_log'] = np.log(master_df['unemployment_rate'])
master_df['payrolls_pct_change'] = master_df['payrolls'].pct_change()

In [ ]:
features_l2 = [
    'payrolls_lag1',
    'log_initial_claims',
    'hire_rate_squared',
    'unemployment_rate_log',
    '10yr_yield',
    'job_opening_rate'
]

X_l2 = master_df[features_l2]
y_l2 = master_df['payrolls'] # MODELING CHANGE IN PAYROLLS RATHER THAN LEVELS TO AVOID NON-STATIONARITY ISSUES

split = int(len(master_df) * 0.8)
X_l2_train = X_l2.iloc[:split]
X_l2_test = X_l2.iloc[split:]
y_l2_train = y_l2.iloc[:split]
y_l2_test = y_l2.iloc[split:]

In [ ]:
model_expectation = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 25), cv=TimeSeriesSplit(n_splits=5)))
])

In [ ]:
model_expectation.fit(X_l2_train, y_l2_train)

y_l2_pred = model_expectation.predict(X_l2_test)

mae = mean_absolute_error(y_l2_test, y_l2_pred)
mse = mean_squared_error(y_l2_test, y_l2_pred)
r2 = r2_score(y_l2_test, y_l2_pred)

The Ridge Regression expectations model achieved an out-of-sample $MAE$ of 4,267 payrolls and an $R^2$ of 0.42. While the model does not fully capture all variance in payroll levels, it explains approximately 42% of the out-of-sample variation. Given the amount of noise we would expect to be present in such a limited macroeconomic dataset, this level of explanatory power is a reasonable result, and acceptable for constructing our surprise variables.

In [ ]:
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"R^2: {r2}")

plt.Figure(figsize=(10,5))

plt.plot(y_l2_test.values, label='Actual')
plt.plot(y_l2_pred, label='Predicted')
plt.legend()
plt.show()

In [ ]:
r2_train = model_expectation.score(X_l2_train, y_l2_train)
r2_test = model_expectation.score(X_l2_test, y_l2_test)
print(f"R^2 (Train): {r2_train}")
print(f"R^2 (Test): {r2_test}")

In [ ]:
master_df['predicted_payrolls'] = model_expectation.predict(X_l2)
master_df['predicted_payrolls_pct_change'] = master_df['predicted_payrolls'].pct_change()

In [ ]:
master_df['predicted_payrolls_pct_change'].describe()

After generating payroll expectations using the Ridge Regression model, payroll surprises were defined as the difference between actual payroll growth and the model's predicted payroll growth. To focus on economically meaningful deviations, a threshold based on the average percentage change in predicted payrolls was applied. Observations exceeding this threshold were classified as upside surprises, while observations falling below the negative threshold were classified as downside surprises.

This process resulted in three distinct categories:

- **Upside Surprise (1):** Actual payroll growth exceeded model expectations by a meaningful margin.
- **No Surprise (0):** Actual payroll growth remained within the threshold band surrounding model expectations.
- **Downside Surprise (-1):** Actual payroll growth fell meaningfully below model expectations.

The resulting distribution of observations was:

| Classification | Observations |
|---------------|-------------:|
| Upside Surprise | 149 |
| No Surprise | 81 |
| Downside Surprise | 55 |

These classifications will serve as the target variables for the models developed in this project. Two approaches will be explored:

1. **Multinomial Classification** – A single model will be trained to classify observations into one of three categories: upside surprise, no surprise, or downside surprise.

2. **Binary Classification** – Separate models will be trained to identify upside surprises and downside surprises independently. This approach allows each model to focus on a specific type of payroll surprise and capture potentially asymmetric relationships between economic indicators and payroll outcomes.

The binary classification approach is particularly appealing in a macroeconomic context because the factors associated with stronger-than-expected payroll growth may differ from those associated with weaker-than-expected payroll growth. By modeling these events separately, the analysis can identify distinct signals that precede upside and downside surprises.

In [ ]:
threshold = 0.001

master_df['upside_surprise'] = master_df['predicted_payrolls_pct_change'].apply(lambda x: 1 if x > threshold else 0)
master_df['downside_surprise'] = master_df['predicted_payrolls_pct_change'].apply(lambda x: 1 if x < -threshold else 0)
master_df['surprise'] = master_df['predicted_payrolls_pct_change'].apply(lambda x: 1 if x > threshold else (-1 if x < -threshold else 0))

In [ ]:
print(master_df['upside_surprise'].value_counts())
print(master_df['downside_surprise'].value_counts())
print(master_df['surprise'].value_counts())

master_df.head()

## **Modeling Upside and Downside Surprises**

Rather than training a single multiclass model, I chose to build two separate XGBoost classifiers: one to predict *upside* payroll surprises and another to predict *downside* surprises. 

By training separate models, each classifier can focus on identifying the specific patterns and nonlinear relationships that precede its target outcome. For example, rising job openings and declining unemployment claims may be strong indicators of an upside surprise, while increasing layoffs and weakening labor demand may be more relevant for predicting downside surprises. Separating these prediction tasks allows the models to learn distinct signal structures and provides a more interpretable assessment of the factors associated with positive and negative labor market surprises.

In [ ]:
features = [
    'sentiment_lag1',
    'payrolls_lag1',
    'payrolls_change',
    'unemployment_rate_3mo_avg',
    'initial_claims_lag1',
    'initial_claims_momentum',
    'cont_claims_lag1',
    'cont_claims_momentum',
    '2yr_yield_volatility',
    '10yr_yield_volatility',
    'jor_3mo_avg',
    'jor_momentum',
    'hire_3mo_avg',
    'hire_momentum',
    'quit_3mo_avg',
    'quit_momentum',
    'discharge_3mo_avg',
    'discharge_momentum'
]

### Upside Surprise Model

In [ ]:
# Upside Surprise Sets

X_upside = master_df[features]
y_upside = master_df['upside_surprise']

X_train_upside, X_test_upside, y_train_upside, y_test_upside = train_test_split(X_upside, y_upside, random_state=42)

In [ ]:
# Initial XGBoost Model for Predicting Upside Surprises

upside_xgb = xgb.XGBClassifier(objective='binary:logistic',
                               eval_metric='auc',
                               seed=42)
upside_xgb.fit(X_train_upside, 
               y_train_upside,
               verbose=False,
               eval_set=[(X_test_upside, y_test_upside)]
               )

The XGBoost classifier achieved a ROC-AUC score of 0.86 and an out-of-sample accuracy of 75% when identifying upside payroll surprises. The model correctly identified 22 of 29 upside surprise observations while maintaining a precision of 67% and recall of 76% for the upside surprise class. The resulting F1 score of 0.71 indicates a strong balance between identifying upside surprise events and limiting false positive signals. The confusion matrix shows that the model correctly classified 54 of 72 observations, including a substantial majority of upside surprise events. 

A key limitation of this analysis is that the surprise variable is defined relative to a model-generated proxy expectation rather than a true market consensus forecast. As a result, the model is effectively predicting deviations from the Ridge Regression forecast rather than deviations from economist expectations or market pricing. While this approach provides a reasonable approximation in the absence of freely available consensus data, the results should be interpreted as evidence of predictive power relative to the proxy expectations model rather than definitive forecasts of actual market surprises.

In [ ]:
upside_xgb_report = classification_report(y_test_upside, upside_xgb.predict(X_test_upside), target_names=['No Upside Surprise', 'Upside Surprise'])
upside_y_probs = upside_xgb.predict_proba(X_test_upside)[:, 1]

print(f"ROC AUC Score for Upside Surprise Model: {roc_auc_score(y_test_upside, upside_y_probs):.4f}")
print("Classification Report for Upside Surprises:\n", upside_xgb_report)

ConfusionMatrixDisplay.from_estimator(upside_xgb, X_test_upside, y_test_upside, display_labels=['No Upside Surprise', 'Upside Surprise'], cmap='Greens')
plt.title('Confusion Matrix for Upside Surprises')
plt.show()

The XGBoost classifier achieved 75% out-of-sample accuracy in identifying upside payroll surprises. The model correctly identified 22 of 29 upside surprise observations while maintaining a precision of 67% and recall of 76%. Results suggest that labor market and financial market indicators contain meaningful predictive information regarding the direction of payroll report surprises, though the model exhibits a tendency to generate some false positive signals.

### Optimized Upside Surprise Model

In [ ]:
# 'reg_lambda': [0, 0.1, 0.5, 1.0]

param_grid = {
    'max_depth': [1, 2, 3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [50, 100, 200, 300],
    'subsample': [0.5, 0.6, 0.8, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.8, 1.0],
    'min_child_weight': [1, 2, 3, 5, 10],
    'gamma': [0, 0.1, 0.3, 0.5, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1.0, 10.0],
    'reg_lambda': [0.1, 1, 10, 100]
}

params = RandomizedSearchCV(
    estimator=upside_xgb,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=TimeSeriesSplit(n_splits=5),
    random_state=42
)

params.fit(X_train_upside, y_train_upside)
print(params.best_params_)
print(params.best_score_)

In [ ]:
# Optimized XGBoost Model for Predicting Upside Surprises

opt_upside_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    **params.best_params_,
    seed=42
)
opt_upside_xgb.fit(X_train_upside, y_train_upside, 
                   verbose=False, 
                   eval_set=[(X_test_upside, y_test_upside)]
                   )

In [ ]:
opt_upside_xgb_report = classification_report(y_test_upside, opt_upside_xgb.predict(X_test_upside), target_names=['No Upside Surprise', 'Upside Surprise'])
opt_upside_y_probs = opt_upside_xgb.predict_proba(X_test_upside)[:, 1]

print(f"ROC AUC Score for Optimized Upside Surprise Model: {roc_auc_score(y_test_upside, opt_upside_y_probs):.4f}")
print("Classification Report for Optimized Upside Surprises:\n", opt_upside_xgb_report)

ConfusionMatrixDisplay.from_estimator(opt_upside_xgb, X_test_upside, y_test_upside, display_labels=['No Upside Surprise', 'Upside Surprise'], cmap='Greens')
plt.title('Confusion Matrix for Optimized Upside Surprise Model')
plt.show()

### Upside ROC Curve

In [ ]:
fpr_opt, tpr_opt, thresholds_opt = roc_curve(
    y_test_upside,
    opt_upside_y_probs
)

plt.figure(figsize=(6,6))
plt.plot(fpr_opt, tpr_opt, label=f"Optimized AUC = {roc_auc_score(y_test_upside, opt_upside_y_probs):.3f}", color='blue')
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test_upside, upside_y_probs):.3f}", color='red')
plt.plot([0,1], [0,1], linestyle="--")
plt.fill_between(fpr_opt, tpr_opt, alpha=0.2, color='blue')
plt.fill_between(fpr, tpr, alpha=0.2, color='red')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Upside Surprise Models")
plt.legend(
    loc='upper right',
    labels=[
        f"CV Optimized AUC = {roc_auc_score(y_test_upside, opt_upside_y_probs):.3f}",
        f"Non-Optimized AUC = {roc_auc_score(y_test_upside, upside_y_probs):.3f}"
    ],
    bbox_to_anchor=(1.6, 1)
)
plt.show()

### Downside Surprise Model

In [ ]:
X_downside = master_df[features]
y_downside = master_df['downside_surprise']

X_train_downside, X_test_downside, y_train_downside, y_test_downside = train_test_split(X_downside, y_downside, random_state=42)

In [ ]:
downside_xgb = xgb.XGBClassifier(objective='binary:logistic',
                                 eval_metric='auc',
                                 seed=42)
downside_xgb.fit(X_train_downside, 
                y_train_downside,
                verbose=False,
                eval_set=[(X_test_downside, y_test_downside)]
                )

In [ ]:
downside_xgb_report = classification_report(y_test_downside, downside_xgb.predict(X_test_downside), target_names=['No Downside Surprise', 'Downside Surprise'])
downside_y_probs = downside_xgb.predict_proba(X_test_downside)[:, 1]

print(f"ROC AUC Score for Optimized Downside Surprise Model: {roc_auc_score(y_test_downside, downside_y_probs):.4f}")
print("Classification Report for Optimized Downside Surprises:\n", downside_xgb_report)

ConfusionMatrixDisplay.from_estimator(downside_xgb, X_test_downside, y_test_downside, display_labels=['No Downside Surprise', 'Downside Surprise'], cmap='Reds')
plt.title('Confusion Matrix for Downside Surprises')
plt.show()

### Optimized Downside Model

In [ ]:
params = RandomizedSearchCV(
    estimator=downside_xgb,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=TimeSeriesSplit(n_splits=5),
    random_state=42
)

params.fit(X_train_downside, y_train_downside)
print(params.best_params_)
print(params.best_score_)

In [ ]:
opt_downside_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    **params.best_params_,
    seed=42
)
opt_downside_xgb.fit(X_train_downside, y_train_downside, verbose=False, eval_set=[(X_test_downside, y_test_downside)])

In [ ]:
opt_downside_xgb_report = classification_report(y_test_downside, opt_downside_xgb.predict(X_test_downside), target_names=['No Downside Surprise', 'Downside Surprise'])
opt_downside_y_probs = opt_downside_xgb.predict_proba(X_test_downside)[:, 1]

print(f"ROC AUC Score for Optimized Downside Surprise Model: {roc_auc_score(y_test_downside, opt_downside_y_probs):.4f}")
print("Classification Report for Optimized Downside Surprises:\n", opt_downside_xgb_report)

ConfusionMatrixDisplay.from_estimator(opt_downside_xgb, X_test_downside, y_test_downside, display_labels=['No Downside Surprise', 'Downside Surprise'], cmap='Reds')
plt.title('Confusion Matrix for Optimized Downside Surprise Model')
plt.show()

### Downside ROC Curve

In [ ]:
opt_fpr, opt_tpr, opt_thresholds = roc_curve(
    y_test_downside,
    opt_downside_y_probs
)

plt.figure(figsize=(6,6))
plt.plot(opt_fpr, opt_tpr, label=f"CV Optimized AUC = {roc_auc_score(y_test_downside, opt_downside_y_probs):.3f}")
plt.plot(fpr, tpr, label=f"Non-Optimized AUC = {roc_auc_score(y_test_downside, downside_y_probs):.3f}")
plt.fill_between(opt_fpr, opt_tpr, alpha=0.2, color='blue')
plt.fill_between(fpr, tpr, alpha=0.2, color='red')
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Downside Surprise Models")
plt.legend(
    loc='upper right',
    labels=[
        f"CV Optimized AUC = {roc_auc_score(y_test_downside, opt_downside_y_probs):.3f}",
        f"Non-Optimized AUC = {roc_auc_score(y_test_downside, downside_y_probs):.3f}"
    ],
    bbox_to_anchor=(1.6, 1)
)
plt.show()